In [ ]:
import python.utils as utils
import python.plots as plots
import importlib

importlib.reload(utils)

X_raw = utils.fetch_dataset_web_traffic('wide')

In [ ]:
X_raw

In [ ]:
importlib.reload(utils)

X = utils.reduce_dataset(X_raw, 300)
X = utils.filter_last_days_available(X, 28*10 + 7)
X = utils.align_fill_and_trim_series(X)

plots.plot_heatmap(X)

In [ ]:
from python.temporal_hierarchy import TemporalHierarchy, SemanticTemporalHierarchy
from python.aggregation import WEEKENDS, WEEKDAYS, WEEKS, FORTNIGHTS, MONTHS, DAYS, ArbitraryAggregation
import importlib
import numpy as np

importlib.reload(plots)

th = SemanticTemporalHierarchy()

summ = th.make_summing_matrix()

plots.plot_summing_matrix(summ)

In [ ]:
from python.temporal_hierarchy import BinaryTemporalHierarchy

bth = BinaryTemporalHierarchy()
s = bth.make_summing_matrix()
plots.plot_summing_matrix(s)


In [ ]:
from python.aggregation import WEEKS, WEEKENDS, WEEKDAYS, FORTNIGHTS, DAYS, MONTHS, ArbitraryAggregation
from python.temporal_hierarchy import  ForecastLevel
from python.time_series import TimeSeriesType
import python.temporal_hierarchy

importlib.reload(python.temporal_hierarchy)

agg = ArbitraryAggregation('28DAYS', np.array(14 * [0, 1]))

fl = ForecastLevel(agg)
fl.prepare_aggregation(X)
fl.make_forecast(28*2)

In [ ]:
fl[TimeSeriesType.BASE_FORECASTS].wide

In [ ]:
from python.aggregation import WEEKS, WEEKENDS, WEEKDAYS, FORTNIGHTS, DAYS, MONTHS, ArbitraryAggregation
from python.temporal_hierarchy import  ForecastLevel,  TemporalHierarchy

th = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, ArbitraryAggregation('Bi', np.array(14 * [0, 1])), DAYS, ])
th.resample_time_series(X)
th.make_base_forecasts(28*2)

In [ ]:
th[WEEKDAYS][TimeSeriesType.BASE_FORECASTS].wide

In [ ]:
th[WEEKDAYS][TimeSeriesType.VALIDATION_SET].wide

In [ ]:
import python.forecast_reconciliation

importlib.reload(python.forecast_reconciliation)

In [ ]:
from python.forecast_reconciliation import ForecastReconciliation

fr = ForecastReconciliation(th)
fr.reconcile(method='shr')

In [ ]:
th[TimeSeriesType.BASE_FORECASTS][0]

In [ ]:
th[TimeSeriesType.RECONCILED_FORECASTS][0]

In [ ]:
import python.error_metrics
importlib.reload(python.error_metrics)

In [ ]:
from python.error_metrics import ErrorMetrics, METRICS, MetricsIndex

em = ErrorMetrics(th)
em.calc_error_metrics()

In [ ]:
em.get_error_stats(MetricsIndex(forecast_type=[TimeSeriesType.BASE_FORECASTS, TimeSeriesType.RECONCILED_FORECASTS], error_metric='mase'))

In [ ]:
importlib.reload(plots)

plots.plot_forecast(th, [DAYS, WEEKS], history_days=1, col=5)

In [ ]:
importlib.reload(plots)

plots.plot_error_distribution(em,
        [MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.RECONCILED_FORECASTS),
         MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.BASE_FORECASTS),
         ])

In [ ]:
plots.plot_error_qq(em,
        MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.RECONCILED_FORECASTS),
        MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.BASE_FORECASTS),
        outlier_threshold=0.95
)

In [ ]:
importlib.reload(plots)

plots.plot_error_diff(em,
        MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.BASE_FORECASTS),
        MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.RECONCILED_FORECASTS),
        outlier_threshold=0.97
)

In [ ]:
plots.plot_error_func(em,
        MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.BASE_FORECASTS),
        MetricsIndex(granularity=DAYS, hierarchy=th, error_metric='mase', forecast_type=TimeSeriesType.RECONCILED_FORECASTS),
        outlier_threshold=0.97
)

In [ ]:
plots.plot_error_distribution(em, MetricsIndex.indeces_from_product(error_metrics=['nmae']))

In [ ]:
a = ArbitraryAggregation.from_summing_matrix(np.array(
    [
        [1, 0, 0, 1] * 7,
        [0, 1, 1, 0] * 7,
        [1, 1, 1, 0, 1, 1, 1] * 4
    ]
))
th_a = TemporalHierarchy(a + [DAYS])
plots.plot_summing_matrix(th_a.make_summing_matrix())

In [ ]:
from python.temporal_hierarchy import SemanticTemporalHierarchy, MinimalTemporalHierarchy
from python.forecast_reconciliation import ForecastReconciliation

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()

sth.resample_time_series(X)
mth.resample_time_series(X)

sth.make_base_forecasts(28*3)
mth.make_base_forecasts(28*3)
mth_shr = mth.clone(clone_suffix='shr')

fr_s = ForecastReconciliation(sth)
fr_m = ForecastReconciliation(mth)
fr_m_s = ForecastReconciliation(mth_shr)


fr_s.reconcile(method='shr')
fr_m_s.reconcile(method='shr')
fr_m.reconcile(method='wls')

# mth_shr.change_reconciled_forecasts_to('shr')
# sth.change_reconciled_forecasts_to('shr') # in case
# mth.change_reconciled_forecasts_to('wls') # just in case

In [ ]:
importlib.reload(python.error_metrics)
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, LoadedErrorMetrics

em = MergedErrorMetrics([
    ErrorMetrics(sth),
    ErrorMetrics(mth),
    ErrorMetrics(mth_shr),
])
em.save('errors.csv')

lem = LoadedErrorMetrics('errors.csv')
lem.metrics